# Análise de Dados - E-commerce Olist

## Tratamento e preparação dos dados

In [1]:
import pandas as pd 

## Carregamento dos dados

Os datasets serão carregados novamente a partir dos arquivos brutos (`raw`). Nesta etapa, os dados ainda não sofrerão alterações e serão utilizados como base para o processo de tratamento.

In [2]:
df_clientes = pd.read_csv("../data/raw/olist_customers_dataset.csv")

df_pedidos = pd.read_csv("../data/raw/olist_orders_dataset.csv")

df_itens = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

df_pagamentos = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

df_avaliacoes = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

df_produtos = pd.read_csv("../data/raw/olist_products_dataset.csv")

df_categorias = pd.read_csv("../data/raw/product_category_name_translation.csv")

## Conversão das colunas de data

As colunas que representam datas estão armazenadas inicialmente como `str`. Nesta etapa, essas colunas serão convertidas para o tipo `datetime`, permitindo a realização de cálculos e análises temporais nas etapas posteriores.

### Conversão das datas de pedidos

As colunas relacionadas às datas dos pedidos serão convertidas para o tipo `datetime`, permitindo análises temporais e cálculos relacionados aos prazos de entrega.

In [3]:
colunas_data_pedidos = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for coluna in colunas_data_pedidos:
    try:
        pd.to_datetime(df_pedidos[coluna])
        print(f"{coluna}: OK")
    except Exception as erro:
        print(f"{coluna}: ERRO")
        print(erro)

order_purchase_timestamp: OK
order_approved_at: OK
order_delivered_carrier_date: OK
order_delivered_customer_date: OK
order_estimated_delivery_date: OK


Como todas as colunas foram validadas com sucesso, elas serão convertidas para o tipo `datetime`.

In [4]:
for coluna in colunas_data_pedidos:
    df_pedidos[coluna] = pd.to_datetime(df_pedidos[coluna])

In [5]:
df_pedidos.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### Conversão da data dos itens

A coluna `shipping_limit_date` representa a data limite de envio dos itens e está armazenada inicialmente como `str`. A coluna será validada e posteriormente convertida para o tipo `datetime`.

In [6]:
df_itens["shipping_limit_date"] = pd.to_datetime(df_itens["shipping_limit_date"])

In [7]:
df_itens.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### Conversão das datas das avaliações

As colunas `review_creation_date` e `review_answer_timestamp` representam datas relacionadas às avaliações dos pedidos e estão armazenadas inicialmente como `str`. As colunas serão convertidas para o tipo `datetime`.

In [8]:
colunas_data_avaliacoes = [
    "review_creation_date",
    "review_answer_timestamp"
]

for coluna in colunas_data_avaliacoes:
    try:
        pd.to_datetime(df_avaliacoes[coluna])
        print(f"{coluna}: OK")
    except Exception as erro:
        print(f"{coluna}: ERRO")
        print(erro)

review_creation_date: OK
review_answer_timestamp: OK


As duas colunas foram validadas com sucesso, elas serão convertidas para o tipo `datetime`.

In [9]:
for coluna in colunas_data_avaliacoes:
    df_avaliacoes[coluna] = pd.to_datetime(df_avaliacoes[coluna])

In [10]:
df_avaliacoes.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

## Tratamento dos valores ausentes

Os valores ausentes identificados durante a exploração inicial serão analisados individualmente antes da aplicação de qualquer tratamento. A estratégia adotada dependerá do significado da ausência e da importância da variável para as análises posteriores.

In [11]:
tabelas = {
    "Clientes": df_clientes,
    "Pedidos": df_pedidos,
    "Itens": df_itens,
    "Pagamentos": df_pagamentos,
    "Avaliações": df_avaliacoes,
    "Produtos": df_produtos,
    "Categorias": df_categorias
}

for nome, df in tabelas.items():
    print(f"{nome}")
    print(df.isna().sum())
    print("-" * 30)
    

Clientes
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
------------------------------
Pedidos
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
------------------------------
Itens
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
------------------------------
Pagamentos
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
------------------------------
Avaliações
review_id           

### Análise dos valores ausentes em pedidos

Os valores ausentes nas colunas relacionadas à aprovação e entrega dos pedidos serão analisados em conjunto com o `order_status`, a fim de verificar se as ausências estão associadas à situação de cada pedido antes de definir uma estratégia de tratamento.

In [12]:
df_pedidos.loc[df_pedidos["order_approved_at"].isna(), "order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [13]:
df_pedidos.loc[df_pedidos["order_delivered_carrier_date"].isna(), "order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [14]:
df_pedidos.loc[df_pedidos["order_delivered_customer_date"].isna(), "order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

#### Decisão de tratamento - Pedidos

Os valores ausentes nas colunas de aprovação e entrega foram analisados em conjunto com o status dos pedidos.

A maior parte das ausências está associada a pedidos que não concluíram determinadas etapas do processo, como pedidos `canceled`, `unavailable`, `processing`, `invoiced` ou `shipped`. Nesses casos, a ausência da respectiva data é compatível com o status do pedido.

Também foram identificados alguns pedidos com status `delivered` que apresentam datas ausentes, indicando registros incompletos na base.

Como não é possível determinar com segurança as datas ausentes, os valores serão mantidos como `NaT`, evitando a criação de informações artificiais ou a exclusão desnecessária de pedidos.

### Análise dos valores ausentes em avaliações

O dataset de avaliações apresenta valores ausentes nas colunas `review_comment_title` e `review_comment_message`. Entretanto, a coluna `review_score` não apresenta valores ausentes, indicando que os registros possuem uma nota de avaliação mesmo quando não há um comentário textual.

Como o preenchimento de título e comentário é opcional e não é possível determinar o conteúdo que seria escrito pelos clientes, os valores ausentes serão mantidos. Essa decisão preserva as avaliações e evita a criação artificial de informações.

### Análise dos valores ausentes em produtos

Os valores ausentes encontrados no dataset de produtos serão investigados antes da definição da estratégia de tratamento, buscando identificar se as ausências estão concentradas nos mesmos registros.

In [15]:
colunas_nulas_produtos = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

df_produtos[colunas_nulas_produtos].isna().all(axis=1).sum()

np.int64(610)

In [16]:
colunas_dimensoes_produtos = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

df_produtos[colunas_dimensoes_produtos].isna().all(axis=1).sum()

np.int64(2)

In [17]:
df_produtos.loc[
    df_produtos["product_weight_g"].isna(),
    [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
produtos_sem_categoria = df_produtos.loc[df_produtos["product_category_name"].isna(), "product_id"]

df_itens["product_id"].isin(produtos_sem_categoria).sum()

np.int64(1603)

In [19]:
df_itens.loc[df_itens["product_id"].isin(produtos_sem_categoria), "product_id"].nunique()

610

#### Decisão de tratamento - Produtos

Os 610 produtos sem categoria foram mantidos, pois todos possuem registros associados na tabela de itens dos pedidos, totalizando 1.603 ocorrências.

Para preservar esses registros nas futuras análises por categoria, os valores ausentes de `product_category_name` serão identificados como `sem_categoria`.

Os valores ausentes relacionados ao tamanho do nome, descrição, quantidade de fotos, peso e dimensões serão mantidos como `NaN`, pois não é possível determinar seus valores reais com segurança e seu preenchimento poderia introduzir informações artificiais nos dados.

In [20]:
df_produtos["product_category_name"] = (df_produtos["product_category_name"].fillna("sem_categoria"))

In [21]:
df_produtos["product_category_name"].isna().sum()

np.int64(0)

## Verificação de inconsistências

Nesta etapa, serão verificadas possíveis inconsistências nos dados, como valores numéricos inválidos e relações temporais incoerentes, antes da preparação final dos datasets.

### Verificação dos valores monetários

Os valores de preço, frete e pagamento serão analisados para identificar possíveis valores negativos ou inconsistentes.

In [22]:
print(f"Menor preço: {df_itens["price"].min()}")
print(f"Menor frete: {df_itens["freight_value"].min()}")
print(f"Menor pagamento: {df_pagamentos["payment_value"].min()}")

Menor preço: 0.85
Menor frete: 0.0
Menor pagamento: 0.0


### Verificação da quantidade de parcelas

A coluna `payment_installments` será analisada para identificar possíveis valores inconsistentes na quantidade de parcelas dos pagamentos.

In [23]:
print("Menor número de parcelas:", df_pagamentos["payment_installments"].min())
print("Maior número de parcelas:", df_pagamentos["payment_installments"].max())

Menor número de parcelas: 0
Maior número de parcelas: 24


In [24]:
df_pagamentos.loc[df_pagamentos["payment_installments"] == 0, ["payment_type", "payment_installments", "payment_value"]]

,payment_type,payment_installments,payment_value
46982,credit_card,0,58.69
79014,credit_card,0,129.94


#### Tratamento das parcelas

Foram identificados dois pagamentos realizados com cartão de crédito (`credit_card`) com `payment_installments` igual a 0.

Como pagamentos com cartão de crédito devem possuir ao menos uma parcela, esses registros foram considerados inconsistentes e o valor será ajustado de 0 para 1.

In [25]:
df_pagamentos.loc[df_pagamentos["payment_installments"] == 0, "payment_installments"] = 1

print("Menor número de parcelas:", df_pagamentos["payment_installments"].min())

Menor número de parcelas: 1


### Verificação das notas de avaliação

A coluna `review_score` será analisada para verificar se as notas registradas estão dentro do intervalo esperado de 1 a 5.

In [26]:
print(f"Nota mínima: {df_avaliacoes['review_score'].min()}")
print(f"Nota máxima: {df_avaliacoes['review_score'].max()}")

Nota mínima: 1
Nota máxima: 5


As notas de avaliação estão dentro do intervalo esperado de 1 a 5, portanto não foram identificadas inconsistências nessa coluna.

### Verificação da consistência das datas

In [27]:
(df_pedidos["order_delivered_customer_date"] < df_pedidos["order_purchase_timestamp"]).sum()

np.int64(0)

In [28]:
(df_pedidos["order_approved_at"] < df_pedidos["order_purchase_timestamp"]).sum()

np.int64(0)

As relações temporais analisadas não apresentaram inconsistências. Não foram identificados pedidos entregues antes da compra ou aprovados antes da realização da compra.

In [29]:
(df_pedidos["order_delivered_carrier_date"] < df_pedidos["order_approved_at"]).sum()

np.int64(1359)

In [30]:
df_pedidos.loc[
    df_pedidos["order_delivered_carrier_date"] < df_pedidos["order_approved_at"],
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28


#### Inconsistências entre aprovação e envio

Foram identificados 1.359 pedidos cuja data de entrega à transportadora é anterior à data de aprovação do pedido.

A análise de alguns desses registros confirmou diferenças que podem chegar a horas ou dias. Como não é possível determinar qual das datas está incorreta nem reconstruir o valor correto com segurança, os registros serão mantidos sem alteração.

Essa inconsistência será considerada em eventuais análises que dependam do intervalo entre aprovação e envio.

In [31]:
(df_pedidos["order_delivered_customer_date"] < df_pedidos["order_delivered_carrier_date"]).sum()

np.int64(23)

In [32]:
df_pedidos.loc[
    df_pedidos["order_delivered_customer_date"] < df_pedidos["order_delivered_carrier_date"],
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01


#### Inconsistências entre envio e entrega

Foram identificados 23 pedidos cuja data de entrega ao cliente é anterior à data de entrega à transportadora.

A análise dos registros confirmou a existência de diferenças de horas ou dias entre essas datas. Como não é possível determinar qual das datas está incorreta nem reconstruir o valor correto com segurança, os registros serão mantidos sem alteração.

Esses casos deverão ser desconsiderados apenas em análises que dependam diretamente do intervalo entre o envio pela transportadora e a entrega ao cliente.

## Padronização dos dados

Nesta etapa, serão realizados ajustes de nomenclatura e consistência para facilitar o uso dos datasets nas análises posteriores.

No dataset de produtos, duas colunas apresentam erro de grafia em seus nomes originais e serão renomeadas.

In [33]:
df_produtos = df_produtos.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)

df_produtos.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [34]:
df_clientes["customer_state"].unique()

<StringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str

In [35]:
df_pedidos["order_status"].unique()

<StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

As colunas categóricas verificadas apresentaram valores padronizados, sem necessidade de alterações adicionais. Os únicos ajustes realizados nesta etapa foram as correções de nomenclatura das colunas `product_name_lenght` e `product_description_lenght`.

## Validação final

Após o tratamento e a padronização dos dados, serão realizadas verificações finais para confirmar que as transformações foram aplicadas corretamente antes do salvamento dos datasets tratados.

In [36]:
print("Categorias ausentes:", df_produtos["product_category_name"].isna().sum())
print("Produtos sem categoria:", (df_produtos["product_category_name"] == "sem_categoria").sum())
print("Menor número de parcelas:", df_pagamentos["payment_installments"].min())

Categorias ausentes: 0
Produtos sem categoria: 610
Menor número de parcelas: 1


In [37]:
print("Pedidos:")
print(df_pedidos[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes)

print("\nItens:")
print(df_itens["shipping_limit_date"].dtype)

print("\nAvaliações:")
print(df_avaliacoes[
    [
        "review_creation_date",
        "review_answer_timestamp"
    ]
].dtypes)

Pedidos:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Itens:
datetime64[us]

Avaliações:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


## Salvamento dos dados tratados

Após a conclusão do tratamento e da validação, os datasets serão salvos em uma pasta separada, preservando os arquivos originais da pasta `data/raw`.

In [39]:
df_clientes.to_csv("../data/processed/clientes.csv", index=False)
df_pedidos.to_csv("../data/processed/pedidos.csv", index=False)
df_itens.to_csv("../data/processed/itens.csv", index=False)
df_pagamentos.to_csv("../data/processed/pagamentos.csv", index=False)
df_avaliacoes.to_csv("../data/processed/avaliacoes.csv", index=False)
df_produtos.to_csv("../data/processed/produtos.csv", index=False)
df_categorias.to_csv("../data/processed/categorias.csv", index=False)